# Prompt Lab — Stage 2 & Stage 4 Prompt Testing

Lightweight notebook for testing prompt variants **without re-running the full agent pipeline**.

**How it works:**
- Cell 2 generates `prompt_test_inputs.json` from the Chicago Sublease PDF (run once, ~30 sec)
- Alternatively, run `agent/agent.ipynb` fully and use its export cell to generate the file from a real lease
- Cells 3–9 load those cached inputs and test 3 prompt variants per stage with a single fast LLM call each
- Each variant is scored against ground truth using the same 8 metrics as `02_evaluate_output.ipynb`
- Final cell recommends the best prompts to copy into `agent/agent.ipynb`

**Stages tested:** Stage 2 (Analyze Clauses) and Stage 4 (Generate Report)  
**Stages fixed:** Stage 1 and Stage 3 prompts are set in `agent/agent.ipynb` and not varied here.

**Prerequisite:** Run `01_build_ground_truth.ipynb` first to generate `ground_truth/ground_truth.json`.

## Cell 1: Setup

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR           = Path("..").resolve()           # final_project/
SECRETS_PATH       = BASE_DIR / "secrets.txt"       # final_project/secrets.txt
GROUND_TRUTH_PATH  = Path("ground_truth/ground_truth.json")
INPUTS_PATH        = Path("prompt_test_inputs.json")

# ── Load API key ────────────────────────────────────────────────────────────
def load_secrets(path=SECRETS_PATH):
    secrets = {}
    try:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    k, v = line.split("=", 1)
                    secrets[k.strip()] = v.strip()
    except FileNotFoundError:
        print(f"secrets.txt not found at {path}. Falling back to environment variable.")
    return secrets

secrets = load_secrets()
api_key = secrets.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Add it to secrets.txt or set as environment variable.")

client = OpenAI(api_key=api_key)
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=api_key)

print("Setup complete.")

# ── Verify input files ──────────────────────────────────────────────────────
for path, label in [
    (GROUND_TRUTH_PATH, "Ground truth"),
    (INPUTS_PATH,       "Prompt test inputs"),
]:
    status = "✓" if Path(path).exists() else "✗ MISSING"
    print(f"  {status}  {label}: {path}")

## Cell 3: Load Cached Inputs

## Cell 2: Generate Test Inputs (run once)

Extracts the Chicago Sublease PDF and runs Stages 1–3 via direct GPT calls to create
`prompt_test_inputs.json`. Skip this cell if the file already exists.

In [ ]:
if INPUTS_PATH.exists():
    print(f"prompt_test_inputs.json already exists — skipping generation.")
    print("Delete the file and re-run this cell to regenerate from scratch.")
else:
    import pypdf

    LEASE_PATH   = BASE_DIR / "rag_data" / "other_leases" / "Chicago Sublease.pdf"
    RENTER_CITY  = "Chicago"
    RENTER_STATE = "Illinois"

    # ── Extract contract text ───────────────────────────────────────────────
    print(f"Extracting text from: {LEASE_PATH.name}")
    reader = pypdf.PdfReader(str(LEASE_PATH))
    contract_text = "\n".join(p.extract_text() or "" for p in reader.pages)
    print(f"  {len(contract_text):,} chars")

    def _call(system_prompt, user_message):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user",   "content": user_message}],
            temperature=0,
        )
        return r.choices[0].message.content

    # ── Stage 1: Define Standards ───────────────────────────────────────────
    print("\nStage 1: Generating standards...")
    _s1_prompt = """You are a tenant rights legal expert. Establish the legal and contractual standards
that apply to a residential lease in the given jurisdiction. Cover:
- Key statutes and local ordinances (security deposit limits, notice requirements, etc.)
- Provisions legally required or prohibited in this jurisdiction
- What fair, balanced language looks like for common clause types
- Red flag language patterns that disadvantage tenants"""
    standards = _call(_s1_prompt,
        f"Location: {RENTER_CITY}, {RENTER_STATE}\n\nLease (first 4000 chars):\n{contract_text[:4000]}")
    print(f"  {len(standards):,} chars")

    # ── Stage 2 baseline: Analyze Clauses (v1_minimal) ─────────────────────
    print("\nStage 2: Analyzing clauses (v1_minimal baseline)...")
    _s2_prompt = """Analyze each clause in this lease. For each clause, classify it as:
illegal, unfair but legal, unclear or ambiguous, outdated, or fair.
Give each clause a severity score from 1 (minor) to 10 (critical)."""
    raw_analysis = _call(_s2_prompt,
        f"Location: {RENTER_CITY}, {RENTER_STATE}\n\nStandards:\n{standards}\n\nLease:\n{contract_text}\n\nAnalyze each clause.")
    print(f"  {len(raw_analysis):,} chars")

    # ── Stage 3: Prioritize Findings ────────────────────────────────────────
    print("\nStage 3: Prioritizing findings...")
    _s3_prompt = """You are a tenant advocate. Rank all non-fair findings by urgency:
- CRITICAL: illegal clauses or severity 8-10
- HIGH: unfair but legal, severity 5-7
- MEDIUM: unclear or ambiguous, severity 3-4
- LOW: severity 1-2
Format: Tier | Clause Name | Why | Negotiable? Skip fair clauses."""
    prioritized = _call(_s3_prompt,
        f"Location: {RENTER_CITY}, {RENTER_STATE}\n\nStandards:\n{standards}\n\nAnalysis:\n{raw_analysis}\n\nRank the findings.")
    print(f"  {len(prioritized):,} chars")

    # ── Save ────────────────────────────────────────────────────────────────
    INPUTS_PATH.write_text(json.dumps({
        "contract_text": contract_text,
        "standards":     standards,
        "raw_analysis":  raw_analysis,
        "prioritized":   prioritized,
        "renter_city":   RENTER_CITY,
        "renter_state":  RENTER_STATE,
    }, indent=2), encoding="utf-8")
    print(f"\nSaved to: {INPUTS_PATH}")
    print("Now run Cell 3 to load these inputs.")

In [ ]:
# ── Ground truth ────────────────────────────────────────────────────────────
with open(GROUND_TRUTH_PATH, encoding="utf-8") as f:
    ground_truth = json.load(f)
gt_clauses = ground_truth["clause_types"]
print(f"Ground truth: {len(gt_clauses)} clause types")

# ── Cached pipeline outputs ─────────────────────────────────────────────────
with open(INPUTS_PATH, encoding="utf-8") as f:
    inputs = json.load(f)

contract_text      = inputs["contract_text"]
standards          = inputs["standards"]
raw_analysis_cache = inputs["raw_analysis"]   # Stage 2 output (baseline)
prioritized_cache  = inputs["prioritized"]    # Stage 3 output (fixed)
RENTER_CITY        = inputs.get("renter_city", "Chicago")
RENTER_STATE       = inputs.get("renter_state", "Illinois")

print(f"Contract text: {len(contract_text):,} chars")
print(f"Standards:     {len(standards):,} chars")
print(f"Location:      {RENTER_CITY}, {RENTER_STATE}")

## Cell 3: Define Stage 2 Prompt Variants

Three variants that progressively add structure and persona — testing one dimension at a time:
- `v1_minimal`: bare instruction, no format enforcement
- `v2_structured`: adds explicit per-clause output format
- `v3_expert`: adds tenant attorney persona on top of structure

In [ ]:
STAGE2_VARIANTS = {
    "v1_minimal": """
Analyze each clause in this lease. For each clause, classify it as:
illegal, unfair but legal, unclear or ambiguous, outdated, or fair.
Give each clause a severity score from 1 (minor) to 10 (critical).
""".strip(),

    "v2_structured": """
You are reviewing a rental lease on behalf of a tenant. Analyze every clause in the lease — do not skip any.

For each clause, output exactly this format:

**Clause**: [clause name or short description]
**Label**: [illegal / unfair but legal / unclear or ambiguous / outdated / fair]
**Severity**: [integer 1-10, where 10 is most harmful to the tenant]
**Explanation**: [1-2 sentences explaining why, citing law or standards where applicable]

Label definitions:
- illegal: violates applicable housing law and is unenforceable
- unfair but legal: legal, but heavily favors the landlord at the tenant's expense
- unclear or ambiguous: vague language that could be misinterpreted or exploited
- outdated: language that no longer reflects current law or practice
- fair: reasonable and balanced

Use the standards framework provided to benchmark each clause. Do not skip any clause.
""".strip(),

    "v3_expert": """
You are a tenant rights attorney with 20 years of experience reviewing residential leases.
A renter has hired you to protect their interests. Your professional reputation depends on catching every problem.

Audit every single clause in this lease. For each clause:
1. Consider what the standards framework says about this type of clause.
2. Ask: does this language protect the tenant, expose them to risk, or is it neutral?
3. Assign a label and severity, then explain your reasoning.

Use exactly this format for each clause:

**Clause**: [clause name or short description]
**Label**: [illegal / unfair but legal / unclear or ambiguous / outdated / fair]
**Severity**: [integer 1-10, where 10 is most harmful to the tenant]
**Explanation**: [1-2 sentences citing specific laws or standard benchmarks]

Label definitions:
- illegal: violates applicable housing law; unenforceable
- unfair but legal: legal but significantly disadvantages the tenant
- unclear or ambiguous: vague or exploitable language
- outdated: no longer reflects current law or practice
- fair: balanced and reasonable

Be exhaustive. Missing a problematic clause is a professional failure.
""".strip(),
}

print(f"Stage 2 variants: {list(STAGE2_VARIANTS.keys())}")

## Cell 4: Define Stage 4 Prompt Variants

In [ ]:
STAGE4_VARIANTS = {
    "v1_minimal": """
Using the clause analysis and prioritized findings, write an Improvement Decision Report
for the tenant summarizing the issues found and what they should do about them.
""".strip(),

    "v2_structured": """
Generate a structured Improvement Decision Report for the tenant based on the clause analysis
and prioritized findings. The report must include these sections:

1. Executive Summary (2-3 sentences: overall verdict on the lease)
2. Key Facts (bullet list of material terms: rent, deposit, term, notice periods)
3. Risk Assessment (all non-fair clauses grouped by CRITICAL / HIGH / MEDIUM / LOW)
4. Prioritized Improvements (for each CRITICAL and HIGH item: the issue, why it matters,
   and a recommended negotiation action)
5. Educational Notes (plain-language explanation of key legal terms mentioned)
6. Citations (laws, ordinances, and standards referenced in the analysis)

Be specific and actionable. Avoid vague advice.
""".strip(),

    "v3_expert": """
You are a tenant rights advocate writing a report for a renter who is not a legal expert.
Your job is to be their champion — clear, empowering, and actionable.

Generate a complete Improvement Decision Report with these sections:

1. **Executive Summary**: 2-3 sentences. Give the tenant a clear bottom line —
   is this lease safe to sign, risky, or problematic?
2. **Key Facts**: Bullet list of material terms (rent, deposit, lease term, notice periods, etc.)
3. **Risk Assessment**: All non-fair clauses grouped by tier (CRITICAL → HIGH → MEDIUM → LOW).
   For each: clause name, label, severity score, one-sentence risk description.
4. **Prioritized Improvements**: For each CRITICAL and HIGH item:
   - The problem in plain language
   - Why it matters financially or legally
   - Exact negotiation language the tenant can use
5. **Educational Notes**: Define legal terms used (joint and several liability, holdover tenant,
   habitability, abatement, etc.) in plain language.
6. **Advocacy Resources**: Note any local tenant rights organizations relevant to the tenant's location.
7. **Citations**: All laws, ordinances, and standards referenced.

Write at an 8th-grade reading level. Tone: empowering, not overwhelming.
""".strip(),
}

print(f"Stage 4 variants: {list(STAGE4_VARIANTS.keys())}")

## Cell 5: Fast Tester & Scorer Functions

In [ ]:
def test_stage2_variant(variant_name, prompt):
    """Run a Stage 2 prompt variant with a single LLM call (no RAG, no tools)."""
    messages = [
        SystemMessage(content=prompt),
        HumanMessage(content=(
            f"Renter's Location: {RENTER_CITY}, {RENTER_STATE}\n\n"
            f"Standards Framework:\n{standards}\n\n"
            f"Lease Contract:\n{contract_text}\n\n"
            "Analyze each clause in this lease."
        )),
    ]
    response = llm.invoke(messages)
    print(f"  [{variant_name}] {len(response.content):,} chars")
    return response.content


def test_stage4_variant(variant_name, prompt, raw_analysis):
    """Run a Stage 4 prompt variant with a single LLM call (no RAG, no tools)."""
    messages = [
        SystemMessage(content=prompt),
        HumanMessage(content=(
            f"Renter's Location: {RENTER_CITY}, {RENTER_STATE}\n\n"
            f"Standards Framework:\n{standards}\n\n"
            f"Clause Analysis:\n{raw_analysis}\n\n"
            f"Prioritized Findings:\n{prioritized_cache}\n\n"
            "Generate the Improvement Decision Report."
        )),
    ]
    response = llm.invoke(messages)
    print(f"  [{variant_name}] {len(response.content):,} chars")
    return response.content


JUDGE_PROMPT_TEMPLATE = """You are an expert evaluator of AI systems that analyze lease agreements for tenant rights.

GROUND TRUTH (clause types a thorough analysis should identify):
{ground_truth}

AGENT OUTPUT BEING EVALUATED:
{agent_output}

Score the output on these 8 dimensions (0-10 each):
1. coverage_rate: Did it identify the key clause types?
2. label_accuracy: Are the labels (illegal/unfair/etc.) correct?
3. accuracy: Are flagged issues genuinely problematic (low false positives)?
4. completeness: Did it miss any significant issues?
5. severity_calibration: Are severity scores appropriate relative to actual risk?
6. recommendation_quality: Are improvement suggestions actionable and legally sound?
7. legal_accuracy: Are legal references and reasoning correct?
8. overall_score: Weighted overall quality.

Return ONLY valid JSON:
{{
  "scores": {{
    "coverage_rate": <0-10>,
    "label_accuracy": <0-10>,
    "accuracy": <0-10>,
    "completeness": <0-10>,
    "severity_calibration": <0-10>,
    "recommendation_quality": <0-10>,
    "legal_accuracy": <0-10>,
    "overall_score": <0-10>
  }},
  "strengths": ["one key strength"],
  "weaknesses": ["one key weakness"]
}}"""


def score_output(agent_output):
    """Score an agent output against ground truth using LLM-as-judge."""
    gt_summary = [
        {
            "name": c["name"],
            "expected_label": c["expected_label"],
            "severity_range": c["severity_range"],
            "what_to_flag": c.get("what_to_flag", ""),
        }
        for c in gt_clauses
    ]
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        ground_truth=json.dumps(gt_summary, indent=2),
        agent_output=agent_output[:4000],
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    return json.loads(response.choices[0].message.content)


print("Functions defined.")

## Cell 6: Test Stage 2 Variants

In [ ]:
print("Testing Stage 2 prompt variants (fast — no RAG)...\n")

stage2_outputs = {}
stage2_scores  = {}
stage2_meta    = {}

for name, prompt in STAGE2_VARIANTS.items():
    print(f"Variant: {name}")
    output = test_stage2_variant(name, prompt)
    stage2_outputs[name] = output

    result = score_output(output)
    stage2_scores[name] = result["scores"]
    stage2_meta[name]   = result
    print(f"  Overall score: {result['scores']['overall_score']}/10")
    print(f"  Strength: {result['strengths'][0] if result.get('strengths') else '—'}")
    print(f"  Weakness: {result['weaknesses'][0] if result.get('weaknesses') else '—'}\n")

print("Done.")

## Cell 7: Stage 2 Comparison Table

In [ ]:
import pandas as pd

rows = []
for name, scores in stage2_scores.items():
    row = {"variant": name}
    row.update(scores)
    rows.append(row)

df_stage2 = pd.DataFrame(rows).set_index("variant")

print("Stage 2 Prompt Variant Scores (out of 10):")
print(df_stage2.to_string())

best_stage2 = df_stage2["overall_score"].idxmax()
print(f"\n→ Best Stage 2 variant: {best_stage2} (overall: {df_stage2.loc[best_stage2, 'overall_score']}/10)")

## Cell 8: Test Stage 4 Variants

Uses the best Stage 2 output as input so Stage 4 is tested on the best available analysis.

In [ ]:
best_stage2_output = stage2_outputs[best_stage2]
print(f"Testing Stage 4 variants using best Stage 2 output ({best_stage2})...\n")

stage4_outputs = {}
stage4_scores  = {}
stage4_meta    = {}

for name, prompt in STAGE4_VARIANTS.items():
    print(f"Variant: {name}")
    output = test_stage4_variant(name, prompt, best_stage2_output)
    stage4_outputs[name] = output

    result = score_output(output)
    stage4_scores[name] = result["scores"]
    stage4_meta[name]   = result
    print(f"  Overall score: {result['scores']['overall_score']}/10")
    print(f"  Strength: {result['strengths'][0] if result.get('strengths') else '—'}")
    print(f"  Weakness: {result['weaknesses'][0] if result.get('weaknesses') else '—'}\n")

print("Done.")

## Cell 9: Stage 4 Comparison Table & Recommendations

In [ ]:
rows = []
for name, scores in stage4_scores.items():
    row = {"variant": name}
    row.update(scores)
    rows.append(row)

df_stage4 = pd.DataFrame(rows).set_index("variant")

print("Stage 4 Prompt Variant Scores (out of 10):")
print(df_stage4.to_string())

best_stage4 = df_stage4["overall_score"].idxmax()
print(f"\n→ Best Stage 4 variant: {best_stage4} (overall: {df_stage4.loc[best_stage4, 'overall_score']}/10)")

print("\n" + "=" * 60)
print(" RECOMMENDATIONS")
print("=" * 60)
print(f"Copy the winning prompts into agent/agent.ipynb cell-3:\n")
print(f"  PROMPT_ANALYZE_CLAUSES = STAGE2_VARIANTS['{best_stage2}']")
print(f"  PROMPT_GENERATE_REPORT = STAGE4_VARIANTS['{best_stage4}']")